%md
# 🧱 Introdução à Arquitetura Medalhão no Databricks

A **Arquitetura Medalhão (Medallion Architecture)** é uma forma de organizar os dados em **camadas progressivas de qualidade** dentro do *Lakehouse* do Databricks.  
O objetivo é garantir que os dados evoluam gradualmente, desde o formato bruto até informações prontas para análise, de forma **modular, auditável e reutilizável**.

---

## 🏗️ Estrutura de Camadas

| Camada | Nome | Objetivo | Nível de Transformação |
|:--|:--|:--|:--|
| 🪶 **Bronze** | Dados Brutos | Armazenar os dados originais, exatamente como vieram da fonte. | Nenhum |
| ⚙️ **Silver** | Dados Tratados | Limpar, tipar e padronizar os dados. | Média |
| 🏆 **Gold** | Dados Analíticos | Consolidar, agregar e modelar métricas e visões de negócio. | Alta |

Cada camada é **independente**, mas **alimenta a próxima**, formando um fluxo de transformação contínuo e transparente.

---

## 🧩 Conceitos Fundamentais no Databricks

Antes de iniciar as camadas da arquitetura, é importante compreender a estrutura básica de armazenamento do Databricks:

### 1️⃣ Catálogo (`Catalog`)
É o **nível mais alto de organização**.  
Agrupa bancos de dados (*schemas*) e controla permissões e governança.  
> Exemplo: `CREATE CATALOG IF NOT EXISTS medalhao;`

---

### 2️⃣ Esquema (`Schema`)
É o equivalente a um **banco de dados** dentro do catálogo.  
Cada camada (bronze, silver, gold) normalmente tem o seu schema.

> Exemplo:  
> `CREATE SCHEMA IF NOT EXISTS bronze;`  
> `CREATE SCHEMA IF NOT EXISTS silver;`  
> `CREATE SCHEMA IF NOT EXISTS gold;`

---

### 3️⃣ Tabela (`Table`)
Contém os dados armazenados em formato **Delta** (tabelas transacionais com versionamento).  
Podem ser criadas a partir de arquivos CSV, Parquet, JSON, etc.

> Exemplo:
> `df.write.format("delta").saveAsTable("bronze.ft_clientes")`

---

### 3️⃣ Volume (`Volume`)
Um espaço de armazenamento de arquivos dentro do Databricks, usado para salvar dados brutos (como CSVs antes da ingestão).

> Exemplo:
> `CREATE VOLUME IF NOT EXISTS landing;
`

%md
# ⚠️ Exclusão completa do Catálogo no Databricks

O comando **`DROP CATALOG`** é utilizado para **remover totalmente um catálogo** do ambiente Databricks, incluindo **todos os schemas, tabelas e views** dentro dele.

---

## 🧹 Comando

> `DROP CATALOG IF EXISTS medalhao CASCADE;`


### Explicação dos parâmetros:

- **IF EXISTS** → evita erro caso o catálogo não exista.
- **CASCADE** → apaga todos os schemas e tabelas associadas.
- **medalhao** → nome do catálogo a ser excluído.

### ⚠️ Atenção

- Esse comando é destrutivo:
- Remove todas as tabelas Bronze, Silver e Gold.
- Apaga todos os metadados e dados físicos associados ao catálogo.
- Não há como desfazer após a execução.

### 💡 Boas práticas

Antes de executar:

- Verifique se há dados importantes a serem mantidos.
- Faça backup ou exportação se necessário.
- Execute apenas em ambientes de desenvolvimento ou teste quando estiver limpando o ciclo de ingestão.

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao;

### 2️⃣ CRIANDO UM SCHEMA
SEÇÕES DA BIBLIOTECA -> Ficção, Aventura...

In [0]:
%sql
USE CATALOG medalhao;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold

### 3️⃣ CRIANDO UMA TABELA
Livros individuais com capítulos estruturados

In [0]:
%sql
USE SCHEMA bronze;
CREATE TABLE IF NOT EXISTS teste_criacao_table

### 4️⃣ CRIANDO UM VOLUME
Revistas, jornais ou outros tipos que não são livros estruturados, mas também faz parte de uma biblioteca

In [0]:
%sql
USE SCHEMA default;
CREATE VOLUME IF NOT EXISTS landing

# 🪶 Camada Bronze – Ingestão de Dados Olist

A **camada Bronze** é a base do *Data Lakehouse*, responsável por **armazenar os dados brutos exatamente como foram recebidos** das fontes originais, sem transformações de negócio que alterem o conteúdo analítico.

É comum acrescentar apenas **metadados de ingestão** (ex.: `timestamp_ingestion`) para auditoria — isso não muda colunas nem regras da fonte.

Ela garante **rastreabilidade e reprocessamento**, funcionando como o registro histórico imutável dos dados.

---

## 🏗️ Tabelas que serão criadas neste notebook

| Arquivo CSV | Tabela Bronze |
|:--|:--|
| olist_customers_dataset.csv | bronze.tb_customers |
| olist_geolocation_dataset.csv | bronze.tb_geolocalizacao |
| olist_order_items_dataset.csv | bronze.tb_order_items |
| olist_order_payments_dataset.csv | bronze.tb_order_payments |
| olist_order_reviews_dataset.csv | bronze.tb_order_reviews |
| olist_orders_dataset.csv | bronze.tb_orders |
| olist_products_dataset.csv | bronze.tb_products |
| olist_sellers_dataset.csv | bronze.tb_sellers |
| product_category_name_translation.csv | bronze.tb_product_category_name_translation |
| API Banco Central | bronze.tb_cotacao_dolar |

## ⚙️ Configuração do Ambiente

Definição do catálogo e schema Bronze que serão utilizados em todo o notebook.

In [0]:
# Definição do catálogo e schema utilizados neste notebook
catalogo = "medalhao"
bronze_db_name = "bronze"

# Caminho do Volume onde os CSVs foram carregados (Landing Zone)
landing_path = "/Volumes/medalhao/default/landing"

print(f"Catálogo: {catalogo}")
print(f"Schema Bronze: {bronze_db_name}")
print(f"Landing Path: {landing_path}")

Catálogo: medalhao
Schema Bronze: bronze
Landing Path: /Volumes/medalhao/default/landing


## 📦 Importações e Função de Ingestão

Criamos uma função reutilizável `ingest_csv` que:
1. Lê o arquivo CSV da Landing Zone
2. Valida se o arquivo não está vazio
3. Adiciona a coluna `timestamp_ingestion` com o momento exato da ingestão
4. Salva como tabela Delta na camada Bronze

In [0]:
from pyspark.sql import functions as F
import requests

# ---------------------------------------------------------------
# Função genérica de ingestão de CSVs para a camada Bronze
# Recebe o nome do arquivo e o nome da tabela de destino
# ---------------------------------------------------------------
def ingest_csv(nome_arquivo, nome_tabela):
    try:
        caminho = f"{landing_path}/{nome_arquivo}"

        # Leitura do CSV com inferência de schema, cabeçalho, multiLine e escape
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("multiLine", "true") # Permite quebra de linha dentro dos textos
            .option("escape", '"')       # Trata aspas duplas corretamente
            .csv(caminho)
        )

        # Validação: verifica se o arquivo não está vazio
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão para rastreabilidade e auditoria
        df_com_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Escrita no formato Delta na camada Bronze
        (
            df_com_metadata.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{catalogo}.{bronze_db_name}.{nome_tabela}")
        )

        print(f"✅ Tabela {bronze_db_name}.{nome_tabela} criada com sucesso! ({df_com_metadata.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao processar {nome_tabela}: {str(e)}\n")

## 🚀 Ingestão das Tabelas CSV

Executamos a ingestão de todos os arquivos CSV do dataset Olist para as tabelas Bronze correspondentes.

In [0]:
# ---------------------------------------------------------------
# Ingestão de todos os arquivos CSV do dataset Olist
# Seguindo o mapeamento definido na atividade
# ---------------------------------------------------------------

ingest_csv("olist_customers_dataset.csv", "tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "tb_order_items")
ingest_csv("olist_order_payments_dataset.csv", "tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "tb_orders")
ingest_csv("olist_products_dataset.csv", "tb_products")
ingest_csv("olist_sellers_dataset.csv", "tb_sellers")
ingest_csv("product_category_name_translation.csv", "tb_product_category_name_translation")

print("🎉 Ingestão dos CSVs da camada Bronze concluída!")

✅ Tabela bronze.tb_customers criada com sucesso! (99441 registros)

✅ Tabela bronze.tb_geolocalizacao criada com sucesso! (1000163 registros)

✅ Tabela bronze.tb_order_items criada com sucesso! (112650 registros)

✅ Tabela bronze.tb_order_payments criada com sucesso! (103886 registros)

✅ Tabela bronze.tb_order_reviews criada com sucesso! (99224 registros)

✅ Tabela bronze.tb_orders criada com sucesso! (99441 registros)

✅ Tabela bronze.tb_products criada com sucesso! (32951 registros)

✅ Tabela bronze.tb_sellers criada com sucesso! (3095 registros)

✅ Tabela bronze.tb_product_category_name_translation criada com sucesso! (71 registros)

🎉 Ingestão dos CSVs da camada Bronze concluída!


In [0]:
# ---------------------------------------------------------------
# Parâmetros do Notebook para configuração das datas da API
# Os widgets permitem que o Job passe parâmetros dinamicamente
# ---------------------------------------------------------------

# Cria widgets de texto para data início e data fim
dbutils.widgets.text("data_inicio", "2017-01-01", "Data Início (AAAA-MM-DD)")
dbutils.widgets.text("data_fim", "2018-12-31", "Data Fim (AAAA-MM-DD)")

# Leitura dos valores dos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data Início: {data_inicio}")
print(f"Data Fim: {data_fim}")

Data Início: 2017-01-01
Data Fim: 2018-12-31


In [0]:
# ---------------------------------------------------------------
# Formatação das datas para o formato esperado pela API do BCB
# Formato exigido: MM-DD-AAAA
# ---------------------------------------------------------------
from datetime import datetime

def formatar_data_bcb(data_str):
    """Converte data de AAAA-MM-DD para MM-DD-AAAA (formato da API do BCB)"""
    dt = datetime.strptime(data_str, "%Y-%m-%d")
    return dt.strftime("%m-%d-%Y")

data_inicio_formatada = formatar_data_bcb(data_inicio)
data_fim_formatada = formatar_data_bcb(data_fim)

print(f"Data Início formatada (MM-DD-AAAA): {data_inicio_formatada}")
print(f"Data Fim formatada (MM-DD-AAAA): {data_fim_formatada}")

Data Início formatada (MM-DD-AAAA): 01-01-2017
Data Fim formatada (MM-DD-AAAA): 12-31-2018


In [0]:
from pyspark.sql import functions as F

# 1. Definição do caminho do arquivo no volume de landing
nome_arquivo_dolar = "cotacao_dolar.csv"
landing_path_dolar = f"/Volumes/medalhao/default/landing/{nome_arquivo_dolar}"

# 2. Leitura do arquivo CSV diretamente pelo Spark
df_dolar = spark.read.csv(landing_path_dolar, header=True, inferSchema=True)

# 3. Adicionando metadados de ingestão (Boas práticas da camada Bronze)
df_dolar = df_dolar.withColumn("fonte_dados", F.lit("API_BCB_MANUAL"))
df_dolar = df_dolar.withColumn("data_ingestao", F.current_timestamp())

# 4. Salvando na camada Bronze no formato Delta
df_dolar.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.tb_cotacao_dolar")

print("✅ Tabela bronze.tb_cotacao_dolar criada com sucesso!")

✅ Tabela bronze.tb_cotacao_dolar criada com sucesso!


## ✅ Verificação Final

Lista todas as tabelas criadas no schema Bronze para confirmar que a ingestão foi bem-sucedida.

In [0]:
# ---------------------------------------------------------------
# Verificação final: lista todas as tabelas criadas no schema Bronze
# ---------------------------------------------------------------
print("📋 Tabelas criadas na camada Bronze:")
spark.sql(f"SHOW TABLES IN {catalogo}.{bronze_db_name}").display()

print("\n🎉 Notebook da Camada Bronze finalizado com sucesso!")

📋 Tabelas criadas na camada Bronze:


database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_customers,false
bronze,tb_geolocalizacao,false
bronze,tb_order_items,false
bronze,tb_order_payments,false
bronze,tb_order_reviews,false
bronze,tb_orders,false
bronze,tb_product_category_name_translation,false
bronze,tb_products,false
bronze,tb_sellers,false



🎉 Notebook da Camada Bronze finalizado com sucesso!
